## GLiNER2: Name evaluation

Датасет: `wolframko/russian-pii-66k` (`source_text` + `privacy_mask`)

- только класс `Name` (gold = `GIVENNAME + SURNAME`)
- метрики на спанах через **посимвольное overlap** (не токены)
- `precision`, `recall`, `f1`
- выборка `8000` примеров из train split (shuffle + fixed seed)

In [ ]:
!pip -q install --no-cache-dir \
    "gliner2" \
    "datasets>=2.20.0" \
    "pandas==2.2.2" \
    "matplotlib" \
    "tqdm"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 44.4 MB/s eta 0:00:00


In [ ]:
import sys
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset

from gliner2 import GLiNER2

print("Python:", sys.version)
print("pandas:", pd.__version__)

In [ ]:
MODEL_NAME = "fastino/gliner2-base-v1"
DATASET_NAME = "wolframko/russian-pii-66k"

TEXT_COL = "source_text"
RAW_MASK_COL = "privacy_mask"
TARGET_LABEL = "Name"

SAMPLE_SIZE = 8000
RANDOM_SEED = 1543

GOLD_NAME_LABELS = {"GIVENNAME", "SURNAME"}
PREDICTION_LABELS = [
    "Name",
    "Person",
    "PER",
    "Full name",
    "Human name",
    "FIO",
    "Имя",
    "ФИО",
]

TARGET_SCHEMA = {
    "target": TARGET_LABEL,
    "gold_raw_labels": sorted(GOLD_NAME_LABELS),
    "prediction_labels": PREDICTION_LABELS,
    "metric_policy": "char_level_overlap",
    "sample_size": SAMPLE_SIZE,
}

print(TARGET_SCHEMA)


In [ ]:
ds = load_dataset(DATASET_NAME, split="train")

if SAMPLE_SIZE is not None:
    sample_n = min(int(SAMPLE_SIZE), len(ds))
    ds = ds.shuffle(seed=RANDOM_SEED).select(range(sample_n))

df = pd.DataFrame({
    TEXT_COL: ds[TEXT_COL],
    RAW_MASK_COL: ds[RAW_MASK_COL],
})

print("shape:", df.shape)
print("columns:", list(df.columns))
print("sample_size_used:", len(df))
display(df.head(3))

In [ ]:
def normalize_span(start, end, text_len):
    if start is None or end is None:
        return None

    start = int(start)
    end = int(end)

    if end <= start:
        return None

    if start < 0:
        start = 0
    if end > text_len:
        end = text_len

    if end <= start:
        return None

    return (start, end)


def merge_spans(spans):
    if not spans:
        return []

    spans = sorted(spans, key=lambda x: (x[0], x[1]))
    merged = [list(spans[0])]

    for start, end in spans[1:]:
        prev_start, prev_end = merged[-1]
        if start <= prev_end:
            merged[-1][1] = max(prev_end, end)
        else:
            merged.append([start, end])

    return [(s, e) for s, e in merged]


def spans_total_len(spans):
    return sum(end - start for start, end in spans)


def spans_intersection_len(a_spans, b_spans):
    i = 0
    j = 0
    inter = 0

    while i < len(a_spans) and j < len(b_spans):
        a_start, a_end = a_spans[i]
        b_start, b_end = b_spans[j]

        left = max(a_start, b_start)
        right = min(a_end, b_end)

        if right > left:
            inter += right - left

        if a_end <= b_end:
            i += 1
        else:
            j += 1

    return inter


def extract_gold_name_spans(text, privacy_mask):
    text_len = len(text)
    spans = []

    for ent in privacy_mask or []:
        if not isinstance(ent, dict):
            continue

        raw_label = str(ent.get("label", "")).upper()
        if raw_label not in GOLD_NAME_LABELS:
            continue

        span = normalize_span(ent.get("start"), ent.get("end"), text_len)
        if span is None:
            continue

        spans.append(span)

    return merge_spans(spans)

In [ ]:
raw_label_counter = Counter()
name_gold_spans_count = 0
empty_name_rows = 0

for row in df.itertuples(index=False):
    text = getattr(row, TEXT_COL)
    privacy_mask = getattr(row, RAW_MASK_COL)

    for ent in privacy_mask or []:
        if isinstance(ent, dict):
            raw_label_counter[str(ent.get("label", "UNKNOWN")).upper()] += 1

    gold_spans = extract_gold_name_spans(text, privacy_mask)
    name_gold_spans_count += len(gold_spans)

    if not gold_spans:
        empty_name_rows += 1

raw_stats = (
    pd.DataFrame(raw_label_counter.items(), columns=["raw_label", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Rows:", len(df))
print("Rows without Name gold spans:", empty_name_rows)
print("Total gold Name spans:", name_gold_spans_count)
display(raw_stats.head(20))

In [ ]:
extractor = GLiNER2.from_pretrained(MODEL_NAME)


def _to_dict(item):
    if isinstance(item, dict):
        return item

    if hasattr(item, "model_dump"):
        try:
            dumped = item.model_dump()
            if isinstance(dumped, dict):
                return dumped
        except Exception:
            pass

    if hasattr(item, "dict"):
        try:
            dumped = item.dict()
            if isinstance(dumped, dict):
                return dumped
        except Exception:
            pass

    as_dict = {}
    for key in [
        "label",
        "entity",
        "entity_group",
        "start",
        "end",
        "start_char",
        "end_char",
        "stop",
        "begin",
        "score",
        "text",
        "span",
    ]:
        if hasattr(item, key):
            as_dict[key] = getattr(item, key)

    return as_dict if as_dict else None


def _extract_list_output(preds):
    if preds is None:
        return []

    if isinstance(preds, dict):
        for key in ["entities", "predictions", "result", "items", "spans"]:
            if key in preds and isinstance(preds[key], list):
                return preds[key]
        return [preds]

    if isinstance(preds, list):
        return preds

    if isinstance(preds, tuple):
        return list(preds)

    return []


def _run_gliner_prediction(text, labels):
    errors = []

    if hasattr(extractor, "predict_entities"):
        try:
            return _extract_list_output(extractor.predict_entities(text, labels)), None
        except Exception as e:
            errors.append(f"predict_entities: {e}")

    if hasattr(extractor, "extract_entities"):
        try:
            return _extract_list_output(extractor.extract_entities(text, labels)), None
        except Exception as e:
            errors.append(f"extract_entities: {e}")

    if callable(extractor):
        try:
            return _extract_list_output(extractor(text, labels=labels)), None
        except Exception as e:
            errors.append(f"callable(labels=): {e}")

        try:
            return _extract_list_output(extractor(text, labels)), None
        except Exception as e:
            errors.append(f"callable(positional): {e}")

    return [], " | ".join(errors) if errors else "No supported prediction method found"


def _span_from_prediction_item(item_dict, text_len):
    for start_key, end_key in [
        ("start", "end"),
        ("start_char", "end_char"),
        ("start", "stop"),
        ("begin", "end"),
    ]:
        if start_key in item_dict and end_key in item_dict:
            return normalize_span(item_dict.get(start_key), item_dict.get(end_key), text_len)

    span_obj = item_dict.get("span")
    if isinstance(span_obj, dict):
        return normalize_span(span_obj.get("start"), span_obj.get("end"), text_len)

    return None


def _is_name_like(label):
    label_low = str(label or "").strip().lower()
    if label_low == "":
        return True

    name_tokens = ["name", "person", "per", "fio", "им", "фио"]
    return any(tok in label_low for tok in name_tokens)


def predict_name_spans(text):
    preds, error = _run_gliner_prediction(text, PREDICTION_LABELS)
    text_len = len(text)

    spans = []
    parsed_items = []

    for raw_item in preds:
        item = _to_dict(raw_item)
        if item is None:
            continue

        span = _span_from_prediction_item(item, text_len)
        if span is None:
            continue

        label = str(item.get("label") or item.get("entity") or item.get("entity_group") or "")
        if not _is_name_like(label):
            continue

        parsed_items.append(
            {
                "label": label,
                "start": span[0],
                "end": span[1],
                "text": text[span[0]:span[1]],
            }
        )
        spans.append(span)

    spans = merge_spans(spans)

    return spans, parsed_items, error

In [ ]:
rows_eval = []

sum_tp_chars = 0
sum_fp_chars = 0
sum_fn_chars = 0
sum_gold_chars = 0
sum_pred_chars = 0

rows_with_pred_spans = 0
rows_with_gold_spans = 0
error_counter = Counter()

for row in tqdm(df.itertuples(index=False), total=len(df), desc="GLiNER2 Name eval"):
    text = getattr(row, TEXT_COL)
    privacy_mask = getattr(row, RAW_MASK_COL)

    gold_spans = extract_gold_name_spans(text, privacy_mask)
    pred_spans, pred_items, pred_error = predict_name_spans(text)

    gold_chars = spans_total_len(gold_spans)
    pred_chars = spans_total_len(pred_spans)
    tp_chars = spans_intersection_len(gold_spans, pred_spans)
    fp_chars = pred_chars - tp_chars
    fn_chars = gold_chars - tp_chars

    if len(gold_spans) > 0:
        rows_with_gold_spans += 1
    if len(pred_spans) > 0:
        rows_with_pred_spans += 1
    if pred_error:
        error_counter[pred_error] += 1

    sum_tp_chars += tp_chars
    sum_fp_chars += fp_chars
    sum_fn_chars += fn_chars
    sum_gold_chars += gold_chars
    sum_pred_chars += pred_chars

    rows_eval.append({
        "text": text,
        "gold_name_spans": gold_spans,
        "pred_name_spans": pred_spans,
        "pred_items": pred_items,
        "pred_error": pred_error,
        "tp_chars": tp_chars,
        "fp_chars": fp_chars,
        "fn_chars": fn_chars,
        "gold_chars": gold_chars,
        "pred_chars": pred_chars,
    })

eval_df = pd.DataFrame(rows_eval)

print("Rows with gold Name spans:", rows_with_gold_spans)
print("Rows with predicted Name spans:", rows_with_pred_spans)
print("Total predicted chars:", sum_pred_chars)

if sum_pred_chars == 0:
    print("WARNING: sum_pred_chars == 0. Top prediction errors:")
    display(pd.DataFrame(error_counter.items(), columns=["error", "count"]).sort_values("count", ascending=False).head(5))

display(eval_df[["text", "gold_name_spans", "pred_name_spans", "tp_chars", "fp_chars", "fn_chars"]].head(5))

In [ ]:
def safe_div(numerator, denominator):
    if denominator == 0:
        return 0.0
    return numerator / denominator


precision = safe_div(sum_tp_chars, sum_tp_chars + sum_fp_chars)
recall = safe_div(sum_tp_chars, sum_tp_chars + sum_fn_chars)
f1 = safe_div(2 * precision * recall, precision + recall)

metrics_df = pd.DataFrame([
    {
        "label": TARGET_LABEL,
        "level": "span_char_overlap",
        "tp_chars": sum_tp_chars,
        "fp_chars": sum_fp_chars,
        "fn_chars": sum_fn_chars,
        "gold_chars": sum_gold_chars,
        "pred_chars": sum_pred_chars,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
])

display(metrics_df)

In [ ]:
def _check_spans_valid(spans, text_len):
    prev_end = -1
    for start, end in spans:
        assert 0 <= start < end <= text_len, (start, end, text_len)
        assert start >= prev_end, (spans, prev_end)
        prev_end = end


for row in eval_df.itertuples(index=False):
    text_len = len(row.text)
    _check_spans_valid(row.gold_name_spans, text_len)
    _check_spans_valid(row.pred_name_spans, text_len)

print("Self-check 1 passed: span structure and bounds are valid.")

In [ ]:
assert sum_tp_chars + sum_fn_chars == sum_gold_chars, (
    sum_tp_chars,
    sum_fn_chars,
    sum_gold_chars,
)

assert sum_tp_chars + sum_fp_chars == sum_pred_chars, (
    sum_tp_chars,
    sum_fp_chars,
    sum_pred_chars,
)

print("Self-check 2 passed: metric identities are consistent.")

In [ ]:
hard_cases_df = eval_df.loc[(eval_df["fp_chars"] > 0) | (eval_df["fn_chars"] > 0)].copy()

hard_cases_df = hard_cases_df.sort_values(
    by=["fn_chars", "fp_chars"],
    ascending=False,
).head(10)

display(
    hard_cases_df[[
        "text",
        "gold_name_spans",
        "pred_name_spans",
        "tp_chars",
        "fp_chars",
        "fn_chars",
        "pred_error",
    ]]
)

print("Self-check 3 done: manual inspection sample built.")

In [ ]:
plot_df = pd.DataFrame([
    {"metric": "precision", "value": float(precision)},
    {"metric": "recall", "value": float(recall)},
    {"metric": "f1", "value": float(f1)},
])

plt.figure(figsize=(6, 3.5))
plt.bar(plot_df["metric"], plot_df["value"], color=["#7db7ff", "#9ce29c", "#ffb870"])
plt.ylim(0, 1)
plt.ylabel("value")
plt.title("GLiNER2 Name: char-level overlap metrics")
plt.grid(axis="y", alpha=0.3)

for idx, value in enumerate(plot_df["value"]):
    plt.text(idx, value + 0.02, f"{value:.4f}", ha="center")

plt.tight_layout()
plt.show()

display(metrics_df)
display(plot_df)